# VayuSwarm — Thermal Classifier (Kaggle)

Trains a **MobileNetV3-Small** thermal image classifier on **real infrared datasets** from HuggingFace.

### Datasets
| Dataset | HuggingFace ID | What it provides |
|---------|---------------|-----------------|
| TIOC — Thermal Images Object Detection | `keremberke/thermal-images-object-detection` | ~4 000 real thermal images, COCO bboxes: **person, car, dog** |
| Thermal Dogs & People | `keremberke/thermal-dogs-and-people-detection` | ~5 000 real FLIR thermal images, COCO bboxes: **person, dog, cat** |

### Output
`best_thermal.pth` + `thermal_classifier.onnx` → auto-pushed to `models/thermal/` in GitHub

### Kaggle Secrets required
`HF_TOKEN` · `GIT_TOKEN`

> Runtime: ~25–35 min on T4 GPU


In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import subprocess
subprocess.check_call(["pip", "install", "-q",
    "torch", "torchvision", "onnx", "onnxruntime",
    "huggingface_hub", "hf-transfer", "Pillow", "pycocotools"])

import os, json, shutil, tempfile, math
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import numpy as np
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── CELL 2: Config & Secrets ─────────────────────────────────────────────────
HF_TOKEN  = os.environ.get("HF_TOKEN", "")
GIT_TOKEN = os.environ.get("GIT_TOKEN", "")

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
except Exception:
    _s = None

if _s:
    try:
        HF_TOKEN = _s.get_secret("HF_TOKEN") or HF_TOKEN
    except Exception as e:
        print(f"⚠ HF_TOKEN secret not found: {e}")
    try:
        GIT_TOKEN = _s.get_secret("GIT_TOKEN") or GIT_TOKEN
    except Exception as e:
        print(f"⚠ GIT_TOKEN secret not found: {e}")

print(f"✅ HF_TOKEN: {'set (' + HF_TOKEN[:8] + '…)' if HF_TOKEN else 'EMPTY'}")
print(f"✅ GIT_TOKEN: {'set (' + GIT_TOKEN[:8] + '…)' if GIT_TOKEN else 'EMPTY'}")

GIT_REPO  = "https://github.com/ved354/swam.git"
GIT_USER  = "ved354"
GIT_EMAIL = "ved354@users.noreply.github.com"

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS     = 40
BATCH_SIZE = 32
LR         = 8e-4
IMG_SIZE   = 224
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "thermal_model"
TIOC_DIR    = WORK_DIR / "tioc"
TDAP_DIR    = WORK_DIR / "tdap"   # thermal-dogs-and-people
DATASET_DIR = WORK_DIR / "thermal_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Classes that ThermalModel must predict
THERMAL_CLASSES = ["background", "human", "vehicle", "animal", "fire"]

print(f"Device : {DEVICE}")
print(f"Classes: {THERMAL_CLASSES}")

In [ ]:
# ── CELL 3: Download Datasets from HuggingFace ───────────────────────────────
from huggingface_hub import snapshot_download
import shutil as _shutil

# ══════════════════════════════════════════════════════════════════════════════
# Nuclear credential cleanup — Kaggle injects HF_TOKEN into os.environ
# and huggingface_hub picks it up even with token=False.
# We must remove it from EVERY possible source.
# ══════════════════════════════════════════════════════════════════════════════

# 1) Remove HF tokens from environment variables
for _env_key in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_TOKEN"]:
    os.environ.pop(_env_key, None)

# 2) Delete cached token file from disk
from pathlib import Path as _P
for _tf in [_P.home() / ".cache" / "huggingface" / "token",
            _P.home() / ".huggingface" / "token"]:
    if _tf.exists():
        _tf.unlink()
        print(f"   🗑 Deleted cached token: {_tf}")

# 3) Call logout() to clear any in-memory session state
try:
    from huggingface_hub import logout
    logout()
except Exception:
    pass

print("✅ All HF credentials cleared — using anonymous access")

# ── Clean up any partial downloads from failed attempts ─────────────────────
for d in [TIOC_DIR, TDAP_DIR]:
    if d.exists() and not any(d.rglob("*.json")):
        _shutil.rmtree(d, ignore_errors=True)
        print(f"   🗑 Removed partial download: {d.name}")

# ── Dataset 1: TIOC — Thermal Images Object Detection ──────────────────────
if not TIOC_DIR.exists() or not any(TIOC_DIR.iterdir()):
    print("📥 Downloading TIOC thermal dataset (person / car / dog)...")
    snapshot_download(
        repo_id="keremberke/thermal-images-object-detection",
        repo_type="dataset",
        local_dir=str(TIOC_DIR),
        token=False,
        max_workers=4,
    )
    print("✅ TIOC downloaded")
else:
    print("✅ TIOC already exists")

# ── Dataset 2: Thermal Dogs and People ──────────────────────────────────────
if not TDAP_DIR.exists() or not any(TDAP_DIR.iterdir()):
    print("📥 Downloading Thermal Dogs & People dataset...")
    snapshot_download(
        repo_id="keremberke/thermal-dogs-and-people-detection",
        repo_type="dataset",
        local_dir=str(TDAP_DIR),
        token=False,
        max_workers=4,
    )
    print("✅ Thermal Dogs & People downloaded")
else:
    print("✅ Thermal Dogs & People already exists")

# Show folder sizes
for name, d in [("TIOC", TIOC_DIR), ("TDAP", TDAP_DIR)]:
    n = sum(1 for _ in d.rglob("*") if _.is_file()) if d.exists() else 0
    print(f"  {name}: {n} files")

In [ ]:
# ── CELL 4: Extract Crops ────────────────────────────────────────────────────
import json, shutil
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image

def extract_tioc_crops(tioc_dir: Path, out_dir: Path, max_per_class: int = 2000):
    """
    TIOC COCO format: train/_annotations.coco.json + train/*.jpg
    Categories: person, car, dog → human, vehicle, animal
    Also extracts background patches from unannotated image regions.
    """
    cat_remap = {
        "person": "human",
        "car":    "vehicle",
        "dog":    "animal",
    }
    counters = {k: 0 for k in ["human", "vehicle", "animal", "background"]}
    rng = np.random.default_rng(7)

    for split in ["train", "valid", "test"]:
        ann_path = tioc_dir / split / "_annotations.coco.json"
        if not ann_path.exists():
            ann_path = next(tioc_dir.glob(f"**/{split}*annotations*.json"), None)
            if ann_path is None:
                continue

        img_dir = ann_path.parent
        with open(ann_path) as f:
            coco = json.load(f)

        cat_id_map = {c["id"]: c["name"].lower() for c in coco.get("categories", [])}
        img_info   = {i["id"]: i for i in coco.get("images", [])}

        # Build per-image annotation list for background extraction
        img_anns: dict = {}
        for ann in coco.get("annotations", []):
            img_anns.setdefault(ann["image_id"], []).append(ann)

        # ── Object crops ──────────────────────────────────────────────────────
        for ann in coco.get("annotations", []):
            cls_name = cat_id_map.get(ann["category_id"], "")
            out_cls  = cat_remap.get(cls_name)
            if not out_cls or counters[out_cls] >= max_per_class:
                continue
            info = img_info.get(ann["image_id"])
            if not info:
                continue
            img_path = img_dir / info["file_name"]
            if not img_path.exists():
                img_path = img_dir / Path(info["file_name"]).name
            if not img_path.exists():
                continue
            try:
                img = Image.open(img_path).convert("L")
                x, y, w, h = [int(v) for v in ann["bbox"]]
                iw, ih = img.size
                pad  = 10
                crop = img.crop((max(0, x-pad), max(0, y-pad),
                                 min(iw, x+w+pad), min(ih, y+h+pad)))
                if crop.size[0] < 12 or crop.size[1] < 12:
                    continue
                dest = out_dir / "train" / out_cls
                dest.mkdir(parents=True, exist_ok=True)
                crop.resize((IMG_SIZE, IMG_SIZE)).save(
                    str(dest / f"tioc_{out_cls}_{counters[out_cls]:05d}.png"))
                counters[out_cls] += 1
            except Exception:
                continue

        # ── Background crops — random patches from unannotated regions ────────
        bg_dest = out_dir / "train" / "background"
        bg_dest.mkdir(parents=True, exist_ok=True)
        for img_id, info in list(img_info.items())[:2000]:
            if counters["background"] >= max_per_class:
                break
            img_path = img_dir / info["file_name"]
            if not img_path.exists():
                img_path = img_dir / Path(info["file_name"]).name
            if not img_path.exists():
                continue
            try:
                img = Image.open(img_path).convert("L")
                iw, ih = img.size
                # Collect annotated bboxes for this image
                ann_boxes = []
                for a in img_anns.get(img_id, []):
                    bx, by, bw, bh = [int(v) for v in a["bbox"]]
                    ann_boxes.append((bx, by, bx+bw, by+bh))

                # Random crops that don't overlap annotated boxes
                patch = 64
                for _ in range(8):
                    if counters["background"] >= max_per_class:
                        break
                    if iw <= patch or ih <= patch:
                        break
                    cx = int(rng.integers(0, iw - patch))
                    cy = int(rng.integers(0, ih - patch))
                    # Check overlap with any annotation
                    overlap = any(
                        cx < x2 and cx+patch > x1 and cy < y2 and cy+patch > y1
                        for x1, y1, x2, y2 in ann_boxes
                    )
                    if overlap:
                        continue
                    crop = img.crop((cx, cy, cx+patch, cy+patch))
                    crop.resize((IMG_SIZE, IMG_SIZE)).save(
                        str(bg_dest / f"bg_{counters['background']:05d}.png"))
                    counters["background"] += 1
            except Exception:
                continue

    for cls, n in counters.items():
        print(f"   TIOC  → {n:4d} {cls}")
    return counters


def extract_tdap_crops(tdap_dir: Path, out_dir: Path, max_per_class: int = 3000):
    """
    keremberke/thermal-dogs-and-people-detection
    COCO format: train/_annotations.coco.json + train/*.jpg
    Categories: person → human, dog/cat → animal
    """
    cat_remap = {
        "person": "human",
        "dog":    "animal",
        "cat":    "animal",
    }
    counters = {k: 0 for k in ["human", "animal"]}

    for split in ["train", "valid", "test"]:
        ann_path = tdap_dir / split / "_annotations.coco.json"
        if not ann_path.exists():
            ann_path = next(tdap_dir.glob(f"**/{split}*annotations*.json"), None)
            if ann_path is None:
                continue

        img_dir = ann_path.parent
        with open(ann_path) as f:
            coco = json.load(f)

        cat_id_map = {c["id"]: c["name"].lower() for c in coco.get("categories", [])}
        img_info   = {i["id"]: i for i in coco.get("images", [])}

        for ann in coco.get("annotations", []):
            cls_name = cat_id_map.get(ann["category_id"], "")
            out_cls  = cat_remap.get(cls_name)
            if not out_cls or counters[out_cls] >= max_per_class:
                continue
            info = img_info.get(ann["image_id"])
            if not info:
                continue
            img_path = img_dir / info["file_name"]
            if not img_path.exists():
                img_path = img_dir / Path(info["file_name"]).name
            if not img_path.exists():
                continue
            try:
                img = Image.open(img_path).convert("L")
                x, y, w, h = [int(v) for v in ann["bbox"]]
                iw, ih = img.size
                pad  = 10
                crop = img.crop((max(0, x-pad), max(0, y-pad),
                                 min(iw, x+w+pad), min(ih, y+h+pad)))
                if crop.size[0] < 12 or crop.size[1] < 12:
                    continue
                dest = out_dir / "train" / out_cls
                dest.mkdir(parents=True, exist_ok=True)
                crop.resize((IMG_SIZE, IMG_SIZE)).save(
                    str(dest / f"tdap_{out_cls}_{counters[out_cls]:05d}.png"))
                counters[out_cls] += 1
            except Exception:
                continue

    for cls, n in counters.items():
        print(f"   TDAP  → {n:4d} {cls}")
    return counters


def gen_fire_samples(bg_dir: Path, fire_dir: Path, n: int = 800):
    """Simulate saturated fire signatures on real thermal backgrounds."""
    fire_dir.mkdir(parents=True, exist_ok=True)
    bgs = list(bg_dir.glob("*.png"))[:300] if bg_dir.exists() else []
    rng = np.random.default_rng(42)
    for i in range(n):
        if bgs:
            base = np.array(Image.open(bgs[i % len(bgs)]).convert("L")
                            .resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)
        else:
            base = rng.normal(35, 8, (IMG_SIZE, IMG_SIZE)).clip(0, 80).astype(np.float32)
        n_blobs = rng.integers(1, 4)
        for _ in range(n_blobs):
            cx = rng.integers(25, IMG_SIZE - 25)
            cy = rng.integers(25, IMG_SIZE - 25)
            r  = rng.integers(6, 28)
            ys, xs = np.ogrid[-r:r+1, -r:r+1]
            mask = (xs**2 + ys**2) <= r**2
            py = np.clip(cy + np.arange(-r, r+1)[:, None], 0, IMG_SIZE - 1)
            px = np.clip(cx + np.arange(-r, r+1)[None, :], 0, IMG_SIZE - 1)
            intensity = 255 - (25 * np.sqrt(xs**2 + ys**2) / r)
            base[py[mask], px[mask]] = np.maximum(base[py[mask], px[mask]], intensity[mask])
        Image.fromarray(base.clip(0, 255).astype(np.uint8), mode="L").save(
            str(fire_dir / f"fire_{i:05d}.png"))
    print(f"   Fire  → {n:4d} physics-accurate samples")


# ── Run extraction ────────────────────────────────────────────────────────────
print("\n📦 Building thermal dataset from real images...")
extract_tioc_crops(TIOC_DIR, DATASET_DIR, max_per_class=2000)
extract_tdap_crops(TDAP_DIR, DATASET_DIR, max_per_class=3000)
gen_fire_samples(DATASET_DIR / "train" / "background",
                 DATASET_DIR / "train" / "fire", n=800)

# ── Val split (20%) ───────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
print("\n🔀 Creating val split (20%)...")
for cls in THERMAL_CLASSES:
    src = DATASET_DIR / "train" / cls
    val = DATASET_DIR / "val"   / cls
    val.mkdir(parents=True, exist_ok=True)
    imgs = list(src.glob("*")) if src.exists() else []
    rng.shuffle(imgs)
    for p in imgs[:max(30, len(imgs) // 5)]:
        shutil.copy2(str(p), str(val / p.name))

print("\n📊 Class sizes (train):")
for cls in THERMAL_CLASSES:
    n = len(list((DATASET_DIR / "train" / cls).glob("*"))) if \
        (DATASET_DIR / "train" / cls).exists() else 0
    print(f"   {cls:12s}: {n}")


In [ ]:
# ── CELL 5: Dataset & DataLoaders ────────────────────────────────────────────
class ThermalDataset(Dataset):
    def __init__(self, root: Path, transform):
        self.samples   = []
        self.transform = transform
        for ci, cls in enumerate(THERMAL_CLASSES):
            d = root / cls
            if d.exists():
                for p in d.glob("*"):
                    self.samples.append((str(p), ci))
        np.random.default_rng(42).shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img = Image.open(self.samples[i][0]).convert("L")
        return self.transform(img), self.samples[i][1]


# Augmentation — thermal-specific (no colour jitter on grayscale)
t_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),      # L → RGB for MobileNet
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.1)), # simulate sensor noise
])
t_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ThermalDataset(DATASET_DIR / "train", t_train)
val_ds   = ThermalDataset(DATASET_DIR / "val",   t_val)
print(f"Dataset: {len(train_ds)} train  {len(val_ds)} val  (real thermal images)")

# Class-balanced sampler weights
counts  = [max(1, len(list((DATASET_DIR/"train"/c).glob("*")))) for c in THERMAL_CLASSES]
cls_wts = torch.tensor([sum(counts)/c for c in counts], dtype=torch.float).to(DEVICE)
print("Class weights:", {c: f"{w:.2f}" for c, w in zip(THERMAL_CLASSES, cls_wts.tolist())})

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)


# ── CELL 5b: Model — MobileNetV3-Small (matches VayuSwarm inference code) ────
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(THERMAL_CLASSES))
model = model.to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MobileNetV3-Small: {params:,} parameters, {len(THERMAL_CLASSES)} classes")

In [ ]:
# ── CELL 6: Training Loop ────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=cls_wts)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
no_improve = 0
PATIENCE = 10

print(f"🚀 Training {EPOCHS} epochs on real TIOC + TDAP data...")

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tc = tt = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)              # single forward pass — reused for both loss and acc
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, pred = out.max(1)           # use same output, no second forward pass
        tt += labels.size(0)
        tc += pred.eq(labels).sum().item()
    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            _, pred = model(imgs).max(1)
            vt += labels.size(0)
            vc += pred.eq(labels).sum().item()

    va = 100.0 * vc / vt
    print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={100.*tc/tt:.1f}%  val={va:.1f}%")

    if va > best_acc:
        best_acc = va
        no_improve = 0
        torch.save(model.state_dict(), str(OUTPUT_DIR / "best_thermal.pth"))
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"⏹ Early stop at epoch {epoch+1}")
            break

print(f"\n✅ Best val accuracy: {best_acc:.1f}% (real thermal images)")


In [ ]:
# ── CELL 7: Export ONNX + Save Metadata ─────────────────────────────────────
model.load_state_dict(torch.load(str(OUTPUT_DIR / "best_thermal.pth"), weights_only=True))
model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy,
    str(OUTPUT_DIR / "thermal_classifier.onnx"),
    input_names=["thermal_image"],
    output_names=["class_probs"],
    dynamic_axes={"thermal_image": {0: "batch"}, "class_probs": {0: "batch"}},
    opset_version=17,
)

metadata = {
    "model":      "vayuswarm_thermal",
    "classes":    THERMAL_CLASSES,
    "input_size": IMG_SIZE,
    "best_val_acc": round(best_acc, 2),
    "training_data": {
        "human":      "keremberke/thermal-images-object-detection + keremberke/thermal-dogs-and-people-detection",
        "vehicle":    "keremberke/thermal-images-object-detection (car)",
        "animal":     "keremberke/thermal-images-object-detection (dog) + thermal-dogs-and-people (dog/cat)",
        "background": "TIOC background patches",
        "fire":       "Physics-accurate fire signatures on real IR backgrounds",
    },
    "train_samples": len(train_ds),
    "val_samples":   len(val_ds),
}
with open(OUTPUT_DIR / "thermal_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Exported: best_thermal.pth  thermal_classifier.onnx  thermal_metadata.json")
print(f"   Val accuracy: {best_acc:.1f}%")
print(f"   Classes: {THERMAL_CLASSES}")

In [ ]:
# ── CELL 8: Push to GitHub ───────────────────────────────────────────────────
import subprocess as _sp

if GIT_TOKEN:
    try:
        auth_url  = GIT_REPO.replace("https://", f"https://{GIT_USER}:{GIT_TOKEN}@")
        clone_dir = Path(tempfile.mkdtemp()) / "swam"
        print(f"\n📤 Cloning {GIT_REPO}...")
        _sp.check_call(["git", "clone", "--depth", "1", auth_url, str(clone_dir)])

        target = clone_dir / "models" / "thermal"
        target.mkdir(parents=True, exist_ok=True)

        for fname in ["best_thermal.pth", "thermal_classifier.onnx", "thermal_metadata.json"]:
            src = OUTPUT_DIR / fname
            if src.exists():
                shutil.copy2(str(src), str(target / fname))
                print(f"   ✅ {fname} ({src.stat().st_size / 1024 / 1024:.1f} MB)")

        env = os.environ.copy()
        for cmd in [
            ["git", "config", "user.name",  GIT_USER],
            ["git", "config", "user.email", GIT_EMAIL],
            ["git", "add", "models/thermal/"],
            ["git", "commit", "-m",
             f"Real-data thermal classifier — val_acc={best_acc:.1f}%, "
             f"datasets: TIOC+TDAP, {len(THERMAL_CLASSES)} classes"],
            ["git", "push", "origin", "main"],
        ]:
            _sp.check_call(cmd, cwd=str(clone_dir), env=env)

        print(f"\n✅ Pushed thermal model to {GIT_REPO}")

    except Exception as e:
        print(f"\n⚠ GitHub push failed: {e}")
        print("  → Download from Kaggle Output tab: thermal_model/")
else:
    print("ℹ No GIT_TOKEN — model saved locally at:", OUTPUT_DIR)

print(f"""
{'='*55}
🎉 Thermal Training Complete!
   Val accuracy : {best_acc:.1f}%
   Datasets     : TIOC + TDAP (keremberke)
   Classes      : {THERMAL_CLASSES}
{'='*55}
""")